In [1]:
import sqlite3

con = sqlite3.connect(':memory:')
cur = con.cursor() 

cur.executescript("""
CREATE TABLE users (id INTEGER, name TEXT);
CREATE TABLE orders (user_id INTEGER, amount INTEGER);
                  
INSERT INTO users VALUES (1, 'anna'), (2, 'boris'), (3, 'vera'), (NULL, 'ghost');
INSERT INTO orders VALUES (1, 100), (1, 200), (2, 50), (NULL, 999), (5, 10);
""")

def q(sql):
    rows = cur.execute(sql).fetchall()
    print(f'{len(rows)} строк:')
    for r in rows: print(' ', r)

In [2]:
q("SELECT * FROM users")
q("SELECT * FROM orders")

4 строк:
  (1, 'anna')
  (2, 'boris')
  (3, 'vera')
  (None, 'ghost')
5 строк:
  (1, 100)
  (1, 200)
  (2, 50)
  (None, 999)
  (5, 10)


In [3]:
# INNER JOIN: SELECT * FROM users JOIN orders ON users.id = orders.user_id -- сколько строк? 
# Что случилось с vera, ghost, заказом 999, заказом user_id=5?
q(
    "SELECT * FROM users JOIN orders ON users.id=orders.user_id"
)
# 3 строки
# vera -- в values нет строки с id = 3 (строка без пары отбрасывается)
# ghost просто не учитываются, так как NULL=NULL возвращает NULL (не TRUE --> не проходит фильтр)
# user_id = 5 -- d users нет строки с id = 5

3 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (2, 'boris', 2, 50)


In [4]:
# LEFT JOIN оттуда же -- сколько строк? Чем заполнены amount у vera?
q(
    "SELECT * FROM users LEFT JOIN orders ON users.id=orders.user_id"
)
# 5 строк (7 после изменений ниже)
# LEFT присоединит полностью левую таблицу не зависимо от того есть совпадение или нет
# В случае отсутвия совпадения к строке левой таблицы будет добавлена NONE, NONE пара
# следовательно у vara amount -- NONE

5 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (2, 'boris', 2, 50)
  (3, 'vera', None, None)
  (None, 'ghost', None, None)


In [5]:
# Обновление таблицы
cur.execute("INSERT INTO users VALUES (1, 'anna_dup')") # дубликат

In [6]:
# Дубликаты с обеих сторон: добавь в users вторую строку с id=1.
# (декартово произведение внутри совпавшего ключа).

# Обновление таблицы
cur.execute("""
DELETE FROM users
WHERE name = 'anna_dup'
  AND rowid NOT IN (SELECT MIN(rowid) FROM users WHERE name = 'anna_dup')
""")

q(
    "SELECT * FROM users JOIN orders ON users.id=orders.user_id"
)

# INNER JOIN: ключ 1: (2 строки в users)*(2 в orders)=4 комбинации (мини-декартово внутри ключа) + boris 1*1 = 5

5 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (2, 'boris', 2, 50)
  (1, 'anna_dup', 1, 100)
  (1, 'anna_dup', 1, 200)


In [7]:
q("SELECT * FROM users")
q("SELECT * FROM orders")

5 строк:
  (1, 'anna')
  (2, 'boris')
  (3, 'vera')
  (None, 'ghost')
  (1, 'anna_dup')
5 строк:
  (1, 100)
  (1, 200)
  (2, 50)
  (None, 999)
  (5, 10)


In [8]:
# CROSS JOIN
q(
    "SELECT * FROM users CROSS JOIN orders"
)
# Это декартово произведение для 2х множеств из 5 элементов --> 5*5=25

25 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (1, 'anna', 2, 50)
  (1, 'anna', None, 999)
  (1, 'anna', 5, 10)
  (2, 'boris', 1, 100)
  (2, 'boris', 1, 200)
  (2, 'boris', 2, 50)
  (2, 'boris', None, 999)
  (2, 'boris', 5, 10)
  (3, 'vera', 1, 100)
  (3, 'vera', 1, 200)
  (3, 'vera', 2, 50)
  (3, 'vera', None, 999)
  (3, 'vera', 5, 10)
  (None, 'ghost', 1, 100)
  (None, 'ghost', 1, 200)
  (None, 'ghost', 2, 50)
  (None, 'ghost', None, 999)
  (None, 'ghost', 5, 10)
  (1, 'anna_dup', 1, 100)
  (1, 'anna_dup', 1, 200)
  (1, 'anna_dup', 2, 50)
  (1, 'anna_dup', None, 999)
  (1, 'anna_dup', 5, 10)


In [9]:
# RIGHT/FULL
q(
    "SELECT * FROM users RIGHT JOIN orders ON users.id=orders.user_id"
)
# 7 строк: 5 сметчатся 2 добавиться так как обязана попасть вся правая таблица + NULL, NULL слева
q(
    "SELECT * FROM users FULL JOIN orders ON users.id=orders.user_id"
)
# FULL -- 9 строк: объединение LEFT и RIGHT

7 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (2, 'boris', 2, 50)
  (1, 'anna_dup', 1, 100)
  (1, 'anna_dup', 1, 200)
  (None, None, None, 999)
  (None, None, 5, 10)
9 строк:
  (1, 'anna', 1, 100)
  (1, 'anna', 1, 200)
  (2, 'boris', 2, 50)
  (3, 'vera', None, None)
  (None, 'ghost', None, None)
  (1, 'anna_dup', 1, 100)
  (1, 'anna_dup', 1, 200)
  (None, None, None, 999)
  (None, None, 5, 10)


In [10]:
# COUNT(*), COUNT(user_id) with LEFT JOIN
q(
    "SELECT COUNT(*), COUNT(user_id)" \
    "FROM users LEFT JOIN orders ON users.id=orders.user_id"
)
# COUNT(*) считает строки, COUNT(col) считает не NULL значения в столбце 

1 строк:
  (7, 5)
